In [1]:
# CELL 0 - Setup
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

pd.set_option("display.float_format", "{:,.3f}".format)   # 3 decimals so rates like 0.975 are not rounded to 0.98
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 80)

DATA_FOLDER = Path("data")   # folder holding portfolio.json, profile.json and transcript.json (relative path = reproducible)

# ASSUMPTION (tested in the data-quality checks): transcript 'time' is in hours and
# portfolio 'duration' is in days, so one day of offer validity = 24 time units.
HOURS_PER_DAY = 24

def readable_table(table_df, percent_columns=(), money_columns=(), count_columns=()):
    """Return a display copy with rates shown as %, money as £ and counts with thousands separators."""
    formatted_df = table_df.copy()
    for column in percent_columns:
        formatted_df[column] = formatted_df[column].map(lambda value: "n/a" if pd.isna(value) else f"{value:.1%}")
    for column in money_columns:
        formatted_df[column] = formatted_df[column].map(lambda value: "n/a" if pd.isna(value) else f"£{value:,.2f}")
    for column in count_columns:
        formatted_df[column] = formatted_df[column].map(lambda value: "n/a" if pd.isna(value) else f"{int(value):,}")
    return formatted_df

In [2]:
# CELL 2 - Load the three raw files (never modified afterwards)
portfolio_raw_df  = pd.read_json(DATA_FOLDER / "portfolio.json",  lines=True)
profile_raw_df    = pd.read_json(DATA_FOLDER / "profile.json",    lines=True)
transcript_raw_df = pd.read_json(DATA_FOLDER / "transcript.json", lines=True)

print("RAW FILES LOADED")
for file_name, raw_df in [("portfolio.json", portfolio_raw_df),
                          ("profile.json", profile_raw_df),
                          ("transcript.json", transcript_raw_df)]:
    print(f"  {file_name:<16} {len(raw_df):>8,} rows x {raw_df.shape[1]} columns | columns: {list(raw_df.columns)}")

RAW FILES LOADED
  portfolio.json         10 rows x 6 columns | columns: ['reward', 'channels', 'difficulty', 'duration', 'offer_type', 'id']
  profile.json       17,000 rows x 5 columns | columns: ['gender', 'age', 'id', 'became_member_on', 'income']
  transcript.json   306,534 rows x 4 columns | columns: ['person', 'event', 'value', 'time']


In [3]:
# CELL 5 - Portfolio: first rows
print("PORTFOLIO (raw) - one row per offer")
display(portfolio_raw_df.head())

PORTFOLIO (raw) - one row per offer


,reward,channels,difficulty,duration,offer_type,id
0,10,"[email, mobile, social]",10,7,bogo,ae264e3637204a6fb9bb56bc8210ddfd
1,10,"[web, email, mobile, social]",10,5,bogo,4d5c57ea9a6940dd891ad53e9dbe8da0
2,0,"[web, email, mobile]",0,4,informational,3f207df678b143eea3cee63160fa8bed
3,5,"[web, email, mobile]",5,7,bogo,9b98b8c7a33c4b65b9aebfe6a799e6d9
4,5,"[web, email]",20,10,discount,0b1e1539f2cc45b7b9fa7c272da2e1d7


In [4]:
# CELL 6 - Portfolio: is the offer id unique?
duplicate_offer_ids = int(portfolio_raw_df["id"].duplicated().sum())
print(f"Duplicate offer ids in portfolio.json: {duplicate_offer_ids} (expected 0, so 'id' is a valid key)")

Duplicate offer ids in portfolio.json: 0 (expected 0, so 'id' is a valid key)


In [5]:
# CELL 7 - Portfolio: working copy with descriptive column names
offer_catalogue_df = portfolio_raw_df.rename(columns={
    "id": "offer_id",
    "reward": "offer_reward_value",      # renamed so it is not confused with the reward PAID on completion
    "difficulty": "offer_min_spend",     # spend a member must reach to complete the offer
    "duration": "offer_duration_days",
}).copy()
print("Offer catalogue columns after renaming:", list(offer_catalogue_df.columns))

Offer catalogue columns after renaming: ['offer_reward_value', 'channels', 'offer_min_spend', 'offer_duration_days', 'offer_type', 'offer_id']


In [6]:
# CELL 8 - Portfolio: one 0/1 flag per advertised channel
# NOTE: an advertised channel is not proof that the channel delivered the offer or caused a response.
for channel_name in ["email", "mobile", "social", "web"]:
    offer_catalogue_df[f"advertised_on_{channel_name}"] = (
        offer_catalogue_df["channels"].apply(lambda channel_list: int(channel_name in channel_list)))
print("ADVERTISED CHANNELS PER OFFER (1 = channel listed)")
display(offer_catalogue_df[["offer_type", "channels", "advertised_on_email", "advertised_on_mobile",
                            "advertised_on_social", "advertised_on_web"]])

ADVERTISED CHANNELS PER OFFER (1 = channel listed)


,offer_type,channels,advertised_on_email,advertised_on_mobile,advertised_on_social,advertised_on_web
0,bogo,"[email, mobile, social]",1,1,1,0
1,bogo,"[web, email, mobile, social]",1,1,1,1
2,informational,"[web, email, mobile]",1,1,0,1
3,bogo,"[web, email, mobile]",1,1,0,1
4,discount,"[web, email]",1,0,0,1
5,discount,"[web, email, mobile, social]",1,1,1,1
6,discount,"[web, email, mobile, social]",1,1,1,1
7,informational,"[email, mobile, social]",1,1,1,0
8,bogo,"[web, email, mobile, social]",1,1,1,1
9,discount,"[web, email, mobile]",1,1,0,1


In [7]:
# CELL 9 - Portfolio: readable offer label instead of the 32-character hash
# Example: BOGO_r10_d10_7d = buy-one-get-one, reward 10, minimum spend 10, valid 7 days.
offer_catalogue_df["offer_label"] = (
    offer_catalogue_df["offer_type"].str.upper().str[:4]
    + "_r" + offer_catalogue_df["offer_reward_value"].astype(str)
    + "_d" + offer_catalogue_df["offer_min_spend"].astype(str)
    + "_" + offer_catalogue_df["offer_duration_days"].astype(int).astype(str) + "d")

In [8]:
# CELL 10 - Portfolio: final catalogue
assert offer_catalogue_df["offer_id"].is_unique, "offer_id must uniquely identify an offer"
assert offer_catalogue_df["offer_label"].is_unique, "each offer label must be unique"
print("OFFER CATALOGUE - 10 offers (4 BOGO, 4 discount, 2 informational)")
display(offer_catalogue_df[["offer_label", "offer_type", "offer_reward_value", "offer_min_spend",
                            "offer_duration_days", "channels"]].sort_values("offer_label"))

OFFER CATALOGUE - 10 offers (4 BOGO, 4 discount, 2 informational)


,offer_label,offer_type,offer_reward_value,offer_min_spend,offer_duration_days,channels
1,BOGO_r10_d10_5d,bogo,10,10,5,"[web, email, mobile, social]"
0,BOGO_r10_d10_7d,bogo,10,10,7,"[email, mobile, social]"
8,BOGO_r5_d5_5d,bogo,5,5,5,"[web, email, mobile, social]"
3,BOGO_r5_d5_7d,bogo,5,5,7,"[web, email, mobile]"
6,DISC_r2_d10_10d,discount,2,10,10,"[web, email, mobile, social]"
9,DISC_r2_d10_7d,discount,2,10,7,"[web, email, mobile]"
5,DISC_r3_d7_7d,discount,3,7,7,"[web, email, mobile, social]"
4,DISC_r5_d20_10d,discount,5,20,10,"[web, email]"
7,INFO_r0_d0_3d,informational,0,0,3,"[email, mobile, social]"
2,INFO_r0_d0_4d,informational,0,0,4,"[web, email, mobile]"


In [9]:
# CELL 12 - Profile: first rows (member id hidden - not needed to read the data)
print("PROFILE (raw) - one row per member")
display(profile_raw_df.drop(columns="id").head())

PROFILE (raw) - one row per member


,gender,age,became_member_on,income
0,NaN,118,20170212,NaN
1,F,55,20170715,"112,000.000"
2,NaN,118,20180712,NaN
3,F,75,20170509,"100,000.000"
4,NaN,118,20170804,NaN


In [10]:
# CELL 13 - Profile: is the member id unique?
duplicate_member_ids = int(profile_raw_df["id"].duplicated().sum())
print(f"Duplicate member ids in profile.json: {duplicate_member_ids} (expected 0, so 'id' is a valid key)")

Duplicate member ids in profile.json: 0 (expected 0, so 'id' is a valid key)


In [11]:
# CELL 14 - Profile: working copy with a descriptive key name
member_profile_df = profile_raw_df.rename(columns={"id": "member_id"}).copy()

In [12]:
# CELL 15 - Profile: convert the membership date from 20170212 (integer) to a real date
member_profile_df["became_member_on"] = pd.to_datetime(
    member_profile_df["became_member_on"].astype(str), format="%Y%m%d", errors="coerce")
print(f"Membership dates range from {member_profile_df['became_member_on'].min():%d %b %Y} "
      f"to {member_profile_df['became_member_on'].max():%d %b %Y}")

Membership dates range from 29 Jul 2013 to 26 Jul 2018


In [13]:
# CELL 16 - Profile: flag incomplete profiles instead of dropping them
# Age 118 always appears together with missing gender and income: it is a placeholder for
# "profile not filled in", not a real age. These members still buy and receive offers, so they stay.
member_profile_df["has_incomplete_profile"] = (
    member_profile_df["age"].eq(118)
    & member_profile_df["gender"].isna()
    & member_profile_df["income"].isna())
member_profile_df["age"] = member_profile_df["age"].where(member_profile_df["age"].ne(118))   # 118 -> missing

# Gender, age and income are kept ONLY for data-quality checks - never for cohorts or targeting
# (customer-experience lead: membership data does not give permission to use them).
incomplete_profile_count = int(member_profile_df["has_incomplete_profile"].sum())
print(f"Incomplete profiles: {incomplete_profile_count:,} of {len(member_profile_df):,} members "
      f"({incomplete_profile_count / len(member_profile_df):.1%}) - flagged, not removed")

Incomplete profiles: 2,175 of 17,000 members (12.8%) - flagged, not removed


In [14]:
# CELL 18 - Transcript: first rows (member id hidden)
print("TRANSCRIPT (raw) - one row per member event; 'value' is a dictionary whose keys depend on the event")
display(transcript_raw_df.drop(columns="person").head())

TRANSCRIPT (raw) - one row per member event; 'value' is a dictionary whose keys depend on the event


,event,value,time
0,offer received,{'offer id': '9b98b8c7a33c4b65b9aebfe6a799e6d9'},0
1,offer received,{'offer id': '0b1e1539f2cc45b7b9fa7c272da2e1d7'},0
2,offer received,{'offer id': '2906b810c7d4411798c6938adc9daaa5'},0
3,offer received,{'offer id': 'fafdcd668e3743c1bb461111dcafc2a4'},0
4,offer received,{'offer id': '4d5c57ea9a6940dd891ad53e9dbe8da0'},0


In [15]:
# CELL 20 - Transcript: which keys appear inside 'value', and for which event type?
keys_by_event_type = (transcript_raw_df.assign(value_keys=transcript_raw_df["value"].apply(lambda d: tuple(sorted(d.keys()))))
                      .groupby("event")["value_keys"].unique())
print("KEYS FOUND INSIDE 'value' FOR EACH EVENT TYPE")
for event_name, key_sets in keys_by_event_type.items():
    print(f"  {event_name:<16} -> {[list(keys) for keys in key_sets]}")
print("Note: received/viewed use 'offer id' (space) but completed uses 'offer_id' (underscore).")

transcript_values_df = pd.json_normalize(transcript_raw_df["value"].tolist())   # one column per dictionary key
for expected_key in ["offer id", "offer_id", "amount", "reward"]:
    if expected_key not in transcript_values_df:
        transcript_values_df[expected_key] = np.nan

KEYS FOUND INSIDE 'value' FOR EACH EVENT TYPE
  offer completed  -> [['offer_id', 'reward']]
  offer received   -> [['offer id']]
  offer viewed     -> [['offer id']]
  transaction      -> [['amount']]
Note: received/viewed use 'offer id' (space) but completed uses 'offer_id' (underscore).


In [16]:
# CELL 21 - Transcript: can the two offer-id keys be safely merged into one column?
rows_with_both_offer_id_keys = int((transcript_values_df["offer id"].notna()
                                    & transcript_values_df["offer_id"].notna()).sum())
print(f"Events carrying BOTH 'offer id' and 'offer_id': {rows_with_both_offer_id_keys} "
      "(0 means the two keys never clash and can be combined)")

Events carrying BOTH 'offer id' and 'offer_id': 0 (0 means the two keys never clash and can be combined)


In [17]:
# CELL 22 - Transcript: flattened event table (grain: one row per member event)
member_events_df = (transcript_raw_df.rename(columns={"person": "member_id", "event": "event_type", "time": "event_hour"})
                    .drop(columns="value"))
member_events_df["offer_id"]           = transcript_values_df["offer id"].combine_first(transcript_values_df["offer_id"]).values
member_events_df["transaction_amount"] = transcript_values_df["amount"].values
member_events_df["reward_paid"]        = transcript_values_df["reward"].values
member_events_df["event_type"]         = member_events_df["event_type"].str.replace(" ", "_")
member_events_df["event_row_number"]   = np.arange(len(member_events_df))   # surrogate key: the file has none

print("EVENTS BY TYPE")
display(member_events_df["event_type"].value_counts().rename("number_of_events").to_frame())
print("FLATTENED EVENT TABLE - first rows (member id hidden)")
display(member_events_df.drop(columns="member_id").head())

EVENTS BY TYPE


,number_of_events
event_type,
transaction,138953
offer_received,76277
offer_viewed,57725
offer_completed,33579


FLATTENED EVENT TABLE - first rows (member id hidden)


,event_type,event_hour,offer_id,transaction_amount,reward_paid,event_row_number
0,offer_received,0,9b98b8c7a33c4b65b9aebfe6a799e6d9,NaN,NaN,0
1,offer_received,0,0b1e1539f2cc45b7b9fa7c272da2e1d7,NaN,NaN,1
2,offer_received,0,2906b810c7d4411798c6938adc9daaa5,NaN,NaN,2
3,offer_received,0,fafdcd668e3743c1bb461111dcafc2a4,NaN,NaN,3
4,offer_received,0,4d5c57ea9a6940dd891ad53e9dbe8da0,NaN,NaN,4


In [18]:
# CELL 24 - Data quality: helper that records one check
def quality_check(check_description, observed_result, expected_result, passed):
    return {"check": check_description, "observed": observed_result,
            "expected": expected_result, "status": "PASS" if passed else "WARN - investigate"}

In [19]:
# CELL 25 - Data quality: inputs for the checks
transactions_df = member_events_df[member_events_df["event_type"] == "transaction"]
offer_events_df = member_events_df[member_events_df["event_type"] != "transaction"]

is_exact_duplicate_event = member_events_df.duplicated(
    subset=["member_id", "event_type", "offer_id", "transaction_amount", "event_hour"], keep="first")

completion_events_df = member_events_df[member_events_df["event_type"] == "offer_completed"].merge(
    offer_catalogue_df[["offer_id", "offer_reward_value", "offer_type"]], on="offer_id", how="left")
completions_with_wrong_reward = int((completion_events_df["reward_paid"] != completion_events_df["offer_reward_value"]).sum())
completions_of_informational_offers = int(completion_events_df["offer_type"].eq("informational").sum())

# A genuine completion should coincide with a purchase by the same member in the same hour.
purchase_member_hours = transactions_df.set_index(["member_id", "event_hour"]).index
completions_without_same_hour_purchase = int(
    (~completion_events_df.set_index(["member_id", "event_hour"]).index.isin(purchase_member_hours)).sum())

offer_send_hours = sorted(int(hour) for hour in
                          member_events_df.loc[member_events_df["event_type"] == "offer_received", "event_hour"].unique())
events_from_unknown_members = int((~member_events_df["member_id"].isin(member_profile_df["member_id"])).sum())
offer_events_with_unknown_offer = int((~offer_events_df["offer_id"].isin(offer_catalogue_df["offer_id"])).sum())
malformed_transactions = int((transactions_df["transaction_amount"].isna() | transactions_df["offer_id"].notna()).sum())
malformed_offer_events = int((offer_events_df["offer_id"].isna() | offer_events_df["transaction_amount"].notna()).sum())
profiles_with_misaligned_missing_fields = int(
    (member_profile_df["age"].isna() != member_profile_df["income"].isna()).sum()
    + (member_profile_df["gender"].isna() != member_profile_df["income"].isna()).sum())
members_without_any_event = int((~member_profile_df["member_id"].isin(member_events_df["member_id"])).sum())

In [20]:
# CELL 26 - Data quality: report
file_row_counts = (len(offer_catalogue_df), len(member_profile_df), len(member_events_df))
data_quality_report_df = pd.DataFrame([
    quality_check("Row counts match the case brief (portfolio / profile / transcript)",
                  " / ".join(f"{n:,}" for n in file_row_counts), "10 / 17,000 / 306,534",
                  file_row_counts == (10, 17000, 306534)),
    quality_check("offer_id and member_id are unique keys",
                  f"{offer_catalogue_df['offer_id'].is_unique} / {member_profile_df['member_id'].is_unique}", "True / True",
                  offer_catalogue_df["offer_id"].is_unique and member_profile_df["member_id"].is_unique),
    quality_check("Events from members not found in profile.json", events_from_unknown_members, 0, events_from_unknown_members == 0),
    quality_check("Offer events whose offer is not in portfolio.json", offer_events_with_unknown_offer, 0, offer_events_with_unknown_offer == 0),
    quality_check("Transactions with no amount, or with an offer id", malformed_transactions, 0, malformed_transactions == 0),
    quality_check("Offer events with no offer id, or with an amount", malformed_offer_events, 0, malformed_offer_events == 0),
    quality_check("Incomplete profiles (age 118 + no gender + no income)", incomplete_profile_count, 2175, incomplete_profile_count == 2175),
    quality_check("Profiles where age, gender and income are NOT missing together",
                  profiles_with_misaligned_missing_fields, 0, profiles_with_misaligned_missing_fields == 0),
    quality_check("Membership dates that failed to convert", int(member_profile_df["became_member_on"].isna().sum()), 0,
                  member_profile_df["became_member_on"].isna().sum() == 0),
    quality_check("Exact duplicate event rows", int(is_exact_duplicate_event.sum()), "0, or explained",
                  is_exact_duplicate_event.sum() == 0),
    quality_check("Transactions with zero or negative amount", int((transactions_df["transaction_amount"] <= 0).sum()), 0,
                  (transactions_df["transaction_amount"] <= 0).sum() == 0),
    quality_check("Completions paying a different reward from the catalogue", completions_with_wrong_reward, 0,
                  completions_with_wrong_reward == 0),
    quality_check("Completions recorded for informational offers (should be impossible)",
                  completions_of_informational_offers, 0, completions_of_informational_offers == 0),
    quality_check("Completions with no purchase by the same member in the same hour",
                  completions_without_same_hour_purchase, 0, completions_without_same_hour_purchase == 0),
    quality_check("Range of the event clock (first - last hour)",
                  f"{member_events_df['event_hour'].min()} - {member_events_df['event_hour'].max()}",
                  "about 30 days if the unit is hours", True),
    quality_check("Hours when offers were sent (multiples of 24 support the 'hours' assumption)",
                  offer_send_hours, "all multiples of 24", all(hour % 24 == 0 for hour in offer_send_hours)),
    quality_check("Members with no events at all", members_without_any_event, "small", True),
])

print("DATA-QUALITY REPORT")
display(data_quality_report_df)
passed_checks = int((data_quality_report_df["status"] == "PASS").sum())
print(f"{passed_checks} of {len(data_quality_report_df)} checks passed.")
print("Duplicate event rows by event type (kept: one purchase can complete two overlapping copies of the same offer):")
display(member_events_df[is_exact_duplicate_event]["event_type"].value_counts().rename("duplicate_rows").to_frame())

DATA-QUALITY REPORT


,check,observed,expected,status
0,Row counts match the case brief (portfolio / profile / transcript),"10 / 17,000 / 306,534","10 / 17,000 / 306,534",PASS
1,offer_id and member_id are unique keys,True / True,True / True,PASS
2,Events from members not found in profile.json,0,0,PASS
3,Offer events whose offer is not in portfolio.json,0,0,PASS
4,"Transactions with no amount, or with an offer id",0,0,PASS
5,"Offer events with no offer id, or with an amount",0,0,PASS
6,Incomplete profiles (age 118 + no gender + no income),2175,2175,PASS
7,"Profiles where age, gender and income are NOT missing together",0,0,PASS
8,Membership dates that failed to convert,0,0,PASS
9,Exact duplicate event rows,397,"0, or explained",WARN - investigate


16 of 17 checks passed.
Duplicate event rows by event type (kept: one purchase can complete two overlapping copies of the same offer):


,duplicate_rows
event_type,
offer_completed,397


## Data structure, joins and filtering decisions

### Grain and keys
| Table | Source | Grain (one row per…) | Key | Rows |
|---|---|---|---|---|
| `offer_catalogue_df` | portfolio.json | offer | `offer_id` (unique) | 10 |
| `member_profile_df` | profile.json | member | `member_id` (unique) | 17,000 |
| `member_events_df` | transcript.json | member event | none in the file, so `event_row_number` was created | 306,534 |
| `offer_instances_df` | built from received events | offer received by a member | `offer_instance_id` | 76,277 |
| `member_summary_df` | built from profile + events | member | `member_id` | 17,000 |

### Event-specific fields inside `value`
| Event type | Count | Keys inside `value` | Becomes column |
|---|---|---|---|
| offer received | 76,277 | `offer id` | `offer_id` |
| offer viewed | 57,725 | `offer id` | `offer_id` |
| offer completed | 33,579 | `offer_id` (underscore), `reward` | `offer_id`, `reward_paid` |
| transaction | 138,953 | `amount` (no offer id) | `transaction_amount` |

The two offer-id spellings never appear in the same event (0 clashes), so they were safely combined into one `offer_id` column.

### Join logic
- Events → offer catalogue: many-to-one on `offer_id`, left join, enforced with `validate="many_to_one"`.
- Events → member profile: many-to-one on `member_id`; all 306,534 events match a profile.
- **Views and completions → offer instances:** matched on member + offer, and only when the event falls inside the offer's validity window (hour received to hour received + duration × 24). A member can receive the same offer more than once, so matching on IDs alone would attach events to the wrong copy.
- Transactions are **not** joined to offers: the data has no field linking a purchase to an offer.

### Filtering decisions
| Decision | Reason |
|---|---|
| No members dropped, including the 2,175 incomplete profiles | They still buy and receive offers; dropping them would remove their spend from every metric |
| Age 118 set to missing and the member flagged `has_incomplete_profile` | 118 is a placeholder, not a real age |
| 397 exact duplicate completion events kept, but reported | One purchase can legitimately complete two overlapping copies of the same offer; the reward total excluding them is shown in the reconciliation as a sensitivity check |
| Informational offers excluded from completion metrics | They have no completion event by design |
| Gender, age and income used only for data-quality checks | No consent to use them for targeting (customer-experience lead) |

### Data-quality result
16 of 17 checks pass; the only warning is the 397 duplicates explained above. Two checks support the assumption that time is measured in hours: every offer was sent at a multiple of 24 (0, 168, 336, 408, 504, 576), and all 33,579 completions fall inside a validity window calculated in hours. Purchases (138,953) and spend (£1,775,451.97) reconcile exactly between the source and the derived tables.

## Data dictionary - derived columns

| Column | Table | Definition |
|---|---|---|
| `offer_label` | offer catalogue | Readable offer name: type_reward_minimum-spend_days, e.g. `BOGO_r10_d10_7d` |
| `offer_reward_value` | offer catalogue | Reward the offer pays on completion (catalogue value) |
| `offer_min_spend` | offer catalogue | Spend needed to complete the offer (raw: `difficulty`) |
| `advertised_on_<channel>` | offer catalogue | 1 if the channel is listed for the offer; not proof of delivery |
| `has_incomplete_profile` | member profile | True when age = 118 and gender and income are missing |
| `event_hour` | member events | Hours since the start of the dataset (assumed unit) |
| `reward_paid` | member events | Reward recorded on a completion event |
| `hour_offer_expires` | offer instances | Hour received + duration in days × 24 |
| `was_viewed` | offer instances | Offer viewed inside its validity window |
| `was_completed` | offer instances | BOGO or discount offer completed inside its validity window |
| `completed_without_prior_view` | offer instances | Completed, but never viewed or viewed only after completion |
| `reward_cost` | offer instances | Offer reward if completed, otherwise 0 |
| `overlaps_identical_earlier_offer` | offer instances | Received while an identical earlier offer to the same member was still valid |
| `spend_during_offer_window` | offer instances | Member's total purchases inside the validity window (context only: other offers may be active) |
| `offers_active_when_received` | offer instances | Offers the member held when this one arrived, including itself |
| `purchase_frequency_cohort` | member summary | F0 = 0, F1 = 1–3, F2 = 4–8, F3 = 9+ purchases in the period |
| `offer_engagement_cohort` | member summary | Share of received offers viewed: E1 under 25%, E2 25–74%, E3 75%+ (E0 = received none) |

Metric definitions, numerators and denominators are in the headline metric sheet (`headline_metrics_df`).

In [21]:
 # CELL 27 - Offer-instance table (grain: one row per offer RECEIVED by a member)
# A member can receive the same offer more than once, so an "offer instance" = (member, offer, hour received).
# Views and completions are matched to the instance whose validity window
# [hour received, hour received + duration x 24] contains them.
offer_instances_df = (
    member_events_df[member_events_df["event_type"] == "offer_received"][["member_id", "offer_id", "event_hour"]]
    .rename(columns={"event_hour": "hour_received"})
    .merge(offer_catalogue_df[["offer_id", "offer_label", "offer_type", "offer_reward_value",
                               "offer_min_spend", "offer_duration_days"]],
           on="offer_id", how="left", validate="many_to_one")
    .reset_index(drop=True))
offer_instances_df["offer_instance_id"] = np.arange(len(offer_instances_df))
offer_instances_df["hour_offer_expires"] = (offer_instances_df["hour_received"]
                                            + offer_instances_df["offer_duration_days"] * HOURS_PER_DAY)

def match_events_to_offer_instances(event_type_name):
    """Return (events inside a matching instance's validity window, events that match no instance)."""
    events_of_this_type = member_events_df[member_events_df["event_type"] == event_type_name][
        ["event_row_number", "member_id", "offer_id", "event_hour"]]
    candidate_matches = offer_instances_df[["offer_instance_id", "member_id", "offer_id",
                                            "hour_received", "hour_offer_expires"]].merge(
        events_of_this_type, on=["member_id", "offer_id"])
    inside_validity_window = ((candidate_matches["event_hour"] >= candidate_matches["hour_received"])
                              & (candidate_matches["event_hour"] <= candidate_matches["hour_offer_expires"]))
    matched_events = candidate_matches[inside_validity_window]
    unmatched_events = events_of_this_type[~events_of_this_type["event_row_number"].isin(matched_events["event_row_number"])]
    return matched_events, unmatched_events

matched_views_df, unmatched_views_df = match_events_to_offer_instances("offer_viewed")
matched_completions_df, unmatched_completions_df = match_events_to_offer_instances("offer_completed")

offer_instances_df["hour_first_viewed"] = offer_instances_df["offer_instance_id"].map(
    matched_views_df.groupby("offer_instance_id")["event_hour"].min())
offer_instances_df["hour_first_completed"] = offer_instances_df["offer_instance_id"].map(
    matched_completions_df.groupby("offer_instance_id")["event_hour"].min())

is_informational_offer = offer_instances_df["offer_type"].eq("informational")
offer_instances_df["was_viewed"] = offer_instances_df["hour_first_viewed"].notna()
offer_instances_df["was_completed"] = offer_instances_df["hour_first_completed"].notna() & ~is_informational_offer
offer_instances_df["completed_after_viewing"] = (
    offer_instances_df["was_completed"] & offer_instances_df["was_viewed"]
    & (offer_instances_df["hour_first_viewed"] <= offer_instances_df["hour_first_completed"]))
# Completed WITHOUT a prior view = reward earned although the member had not seen the offer first,
# so the offer could not have influenced that purchase (as far as view tracking shows).
offer_instances_df["completed_without_prior_view"] = (
    offer_instances_df["was_completed"] & ~offer_instances_df["completed_after_viewing"])
offer_instances_df["reward_cost"] = np.where(offer_instances_df["was_completed"],
                                             offer_instances_df["offer_reward_value"], 0)

# Flag instances received while an identical earlier offer to the same member was still valid.
offer_instances_df = offer_instances_df.sort_values(["member_id", "offer_id", "hour_received"])
expiry_of_previous_identical_offer = offer_instances_df.groupby(["member_id", "offer_id"])["hour_offer_expires"].shift()
offer_instances_df["overlaps_identical_earlier_offer"] = offer_instances_df["hour_received"] <= expiry_of_previous_identical_offer
offer_instances_df = offer_instances_df.sort_values("offer_instance_id")

# Member spend while the offer was valid. CONTEXT ONLY: other offers may be active at the same time.
purchases_during_offer_window_df = offer_instances_df[["offer_instance_id", "member_id", "hour_received", "hour_offer_expires"]].merge(
    transactions_df[["member_id", "event_hour", "transaction_amount"]], on="member_id")
purchases_during_offer_window_df = purchases_during_offer_window_df[
    (purchases_during_offer_window_df["event_hour"] >= purchases_during_offer_window_df["hour_received"])
    & (purchases_during_offer_window_df["event_hour"] <= purchases_during_offer_window_df["hour_offer_expires"])]
offer_instances_df["spend_during_offer_window"] = offer_instances_df["offer_instance_id"].map(
    purchases_during_offer_window_df.groupby("offer_instance_id")["transaction_amount"].sum()).fillna(0)

# How many offers (including this one) the member held when this offer arrived.
offer_pairs_same_member_df = offer_instances_df[["offer_instance_id", "member_id", "hour_received"]].merge(
    offer_instances_df[["member_id", "hour_received", "hour_offer_expires"]].rename(
        columns={"hour_received": "other_offer_received", "hour_offer_expires": "other_offer_expires"}),
    on="member_id")
offers_active_at_same_time_df = offer_pairs_same_member_df[
    (offer_pairs_same_member_df["other_offer_received"] <= offer_pairs_same_member_df["hour_received"])
    & (offer_pairs_same_member_df["hour_received"] <= offer_pairs_same_member_df["other_offer_expires"])]
offer_instances_df["offers_active_when_received"] = offer_instances_df["offer_instance_id"].map(
    offers_active_at_same_time_df.groupby("offer_instance_id").size())

completion_events_matched_to_several_instances = int(
    (matched_completions_df.groupby("event_row_number").size() > 1).sum())

print("EVENT-SEQUENCE DIAGNOSTICS")
display(pd.Series({
    "Offer instances (offers received)": len(offer_instances_df),
    "Views that fall outside every validity window": len(unmatched_views_df),
    "Completions that fall outside every validity window": len(unmatched_completions_df),
    "Completion events matched to more than one instance": completion_events_matched_to_several_instances,
    "Instances received while an identical offer was still valid": int(offer_instances_df["overlaps_identical_earlier_offer"].sum()),
    "Completed instances that were never viewed": int((offer_instances_df["was_completed"] & ~offer_instances_df["was_viewed"]).sum()),
    "Completed instances viewed only AFTER completion": int((offer_instances_df["was_completed"] & offer_instances_df["was_viewed"]
                                                             & ~offer_instances_df["completed_after_viewing"]).sum()),
    "Median number of offers a member held at once": int(offer_instances_df["offers_active_when_received"].median()),
}, name="count", dtype=object).to_frame())

EVENT-SEQUENCE DIAGNOSTICS


,count
Offer instances (offers received),76277
Views that fall outside every validity window,1158
Completions that fall outside every validity window,0
Completion events matched to more than one instance,1776
Instances received while an identical offer was still valid,5295
Completed instances that were never viewed,5734
Completed instances viewed only AFTER completion,4220
Median number of offers a member held at once,2


In [22]:
# CELL 28 - Member summary (grain: one row per member) + behavioural cohorts
purchases_per_member_df = transactions_df.groupby("member_id").agg(
    number_of_purchases=("transaction_amount", "size"),
    total_spend=("transaction_amount", "sum")).reset_index()
offer_activity_per_member_df = offer_instances_df.groupby("member_id").agg(
    offers_received=("offer_instance_id", "size"),
    offers_viewed=("was_viewed", "sum"),
    offers_completed=("was_completed", "sum"),
    offers_completed_without_view=("completed_without_prior_view", "sum"),
    reward_cost=("reward_cost", "sum")).reset_index()

member_summary_df = (member_profile_df[["member_id", "has_incomplete_profile"]]
                     .merge(purchases_per_member_df, on="member_id", how="left", validate="one_to_one")
                     .merge(offer_activity_per_member_df, on="member_id", how="left", validate="one_to_one"))
activity_columns = ["number_of_purchases", "total_spend", "offers_received", "offers_viewed",
                    "offers_completed", "offers_completed_without_view", "reward_cost"]
member_summary_df[activity_columns] = member_summary_df[activity_columns].fillna(0)   # no activity = 0, not missing
whole_number_columns = [column for column in activity_columns if column not in ("total_spend", "reward_cost")]
member_summary_df[whole_number_columns] = member_summary_df[whole_number_columns].astype(int)

# Behavioural cohorts only - privacy-safe, no demographics.
member_summary_df["purchase_frequency_cohort"] = pd.cut(
    member_summary_df["number_of_purchases"], bins=[-1, 0, 3, 8, np.inf],
    labels=["F0 no purchases", "F1 1-3 purchases", "F2 4-8 purchases", "F3 9+ purchases"])
share_of_offers_viewed = member_summary_df["offers_viewed"] / member_summary_df["offers_received"].replace(0, np.nan)
member_summary_df["offer_engagement_cohort"] = np.select(
    [member_summary_df["offers_received"].eq(0), share_of_offers_viewed >= 0.75, share_of_offers_viewed >= 0.25],
    ["E0 received no offers", "E3 views most offers (75%+)", "E2 views some offers (25-74%)"],
    default="E1 rarely views offers (<25%)")

offer_instances_df = offer_instances_df.merge(
    member_summary_df[["member_id", "purchase_frequency_cohort", "offer_engagement_cohort"]],
    on="member_id", how="left", validate="many_to_one")

print("MEMBERS PER PURCHASE-FREQUENCY COHORT (number of purchases in the dataset period)")
display(member_summary_df["purchase_frequency_cohort"].value_counts().sort_index().rename("members").to_frame())
print("MEMBERS PER OFFER-ENGAGEMENT COHORT (share of received offers the member viewed)")
display(member_summary_df["offer_engagement_cohort"].value_counts().sort_index().rename("members").to_frame())

MEMBERS PER PURCHASE-FREQUENCY COHORT (number of purchases in the dataset period)


,members
purchase_frequency_cohort,
F0 no purchases,422
F1 1-3 purchases,2501
F2 4-8 purchases,7265
F3 9+ purchases,6812


MEMBERS PER OFFER-ENGAGEMENT COHORT (share of received offers the member viewed)


,members
offer_engagement_cohort,
E0 received no offers,6
E1 rarely views offers (<25%),428
E2 views some offers (25-74%),6311
E3 views most offers (75%+),10255


In [23]:
# CELL 29 - Headline metric sheet (every metric shows its numerator and denominator)
def build_metric(metric_name, numerator, denominator, value_format, definition):
    metric_value = numerator / denominator if denominator else np.nan
    readable_value = f"{metric_value:.1%}" if value_format == "percent" else f"£{metric_value:,.2f}"
    return {"metric": metric_name, "numerator": f"{numerator:,.2f}" if isinstance(numerator, float) else f"{numerator:,}",
            "denominator": f"{denominator:,}", "value": readable_value, "definition": definition}

total_members = len(member_summary_df)
purchasing_members = int((member_summary_df["number_of_purchases"] > 0).sum())
total_member_spend = float(transactions_df["transaction_amount"].sum())
total_purchases = len(transactions_df)
promotional_offer_instances_df = offer_instances_df[offer_instances_df["offer_type"] != "informational"]
completed_promotional_offers = int(promotional_offer_instances_df["was_completed"].sum())
recorded_reward_face_value = float(member_events_df.loc[member_events_df["event_type"] == "offer_completed", "reward_paid"].sum())

headline_metrics_df = pd.DataFrame([
    build_metric("Purchasing-member rate", purchasing_members, total_members, "percent",
                 "members with at least 1 purchase / all members in profile"),
    build_metric("Spend per member", total_member_spend, total_members, "money",
                 "total purchase amount / all members in profile"),
    build_metric("Spend per purchasing member", total_member_spend, purchasing_members, "money",
                 "total purchase amount / members with at least 1 purchase"),
    build_metric("Average purchase value", total_member_spend, total_purchases, "money",
                 "total purchase amount / number of purchases"),
    build_metric("Offer view rate (all offer types)", int(offer_instances_df["was_viewed"].sum()), len(offer_instances_df), "percent",
                 "offers viewed while valid / offers received"),
    build_metric("Offer completion rate (BOGO + discount)", completed_promotional_offers, len(promotional_offer_instances_df), "percent",
                 "offers completed while valid / BOGO + discount offers received"),
    build_metric("Completions with no prior view", int(promotional_offer_instances_df["completed_without_prior_view"].sum()),
                 completed_promotional_offers, "percent", "completed offers not viewed before completion / completed offers"),
    build_metric("Reward face value as a share of spend", recorded_reward_face_value, total_member_spend, "percent",
                 "reward on all completion events / total purchase amount"),
])
print("HEADLINE METRICS - one dataset period (about 30 days, treated as one campaign cycle)")
display(headline_metrics_df)

HEADLINE METRICS - one dataset period (about 30 days, treated as one campaign cycle)


,metric,numerator,denominator,value,definition
0,Purchasing-member rate,"16,578","17,000",97.5%,members with at least 1 purchase / all members in profile
1,Spend per member,"1,775,451.97","17,000",£104.44,total purchase amount / all members in profile
2,Spend per purchasing member,"1,775,451.97","16,578",£107.10,total purchase amount / members with at least 1 purchase
3,Average purchase value,"1,775,451.97","138,953",£12.78,total purchase amount / number of purchases
4,Offer view rate (all offer types),"56,895","76,277",74.6%,offers viewed while valid / offers received
5,Offer completion rate (BOGO + discount),"33,631","61,042",55.1%,offers completed while valid / BOGO + discount offers received
6,Completions with no prior view,"9,954","33,631",29.6%,completed offers not viewed before completion / completed offers
7,Reward face value as a share of spend,"164,676.00","1,775,451.97",9.3%,reward on all completion events / total purchase amount


In [24]:
# CELL 30 - Comparisons: offer type, individual offer, behavioural cohorts, channel
def summarise_offer_performance(offer_instances_subset_df, group_by_columns):
    grouped_instances = offer_instances_subset_df.groupby(group_by_columns, observed=True)
    performance_df = pd.DataFrame({
        "offers_received":               grouped_instances.size(),
        "members_reached":               grouped_instances["member_id"].nunique(),
        "view_rate":                     grouped_instances["was_viewed"].mean(),
        "completion_rate":               grouped_instances["was_completed"].sum() / grouped_instances.size(),
        "completed_without_view_share":  grouped_instances["completed_without_prior_view"].sum()
                                         / grouped_instances["was_completed"].sum().replace(0, np.nan),
        "reward_cost":                   grouped_instances["reward_cost"].sum(),
        "reward_cost_per_offer_sent":    grouped_instances["reward_cost"].sum() / grouped_instances.size(),
        "reward_cost_without_view":      grouped_instances.apply(
            lambda group: group.loc[group["completed_without_prior_view"], "reward_cost"].sum(), include_groups=False),
        "avg_spend_during_offer_window": grouped_instances["spend_during_offer_window"].mean(),
    })
    # Informational offers cannot be completed, so completion metrics do not apply to them.
    is_informational_row = performance_df.index.get_level_values(-1).astype(str).str.contains("informational|INFO")
    performance_df.loc[is_informational_row, ["completion_rate", "completed_without_view_share"]] = np.nan
    return performance_df

def readable_offer_performance(performance_df):
    return readable_table(performance_df,
                          percent_columns=["view_rate", "completion_rate", "completed_without_view_share"],
                          money_columns=["reward_cost", "reward_cost_per_offer_sent", "reward_cost_without_view",
                                         "avg_spend_during_offer_window"],
                          count_columns=["offers_received", "members_reached"])

performance_by_offer_type_df = summarise_offer_performance(offer_instances_df, "offer_type")
performance_by_offer_df = summarise_offer_performance(offer_instances_df, ["offer_type", "offer_label"])
performance_by_engagement_cohort_df = summarise_offer_performance(promotional_offer_instances_df, "offer_engagement_cohort")
performance_by_frequency_cohort_df = summarise_offer_performance(promotional_offer_instances_df, "purchase_frequency_cohort")

# Association only - an advertised channel is not proof of delivery or cause.
view_rate_by_social_channel_df = (
    offer_instances_df.merge(offer_catalogue_df[["offer_id", "advertised_on_social"]], on="offer_id")
    .groupby(["offer_type", "advertised_on_social"])["was_viewed"].mean().unstack()
    .rename(columns={0: "view_rate_not_on_social", 1: "view_rate_on_social"}))
view_rate_by_social_channel_df.columns.name = None

spend_by_frequency_cohort_df = member_summary_df.groupby("purchase_frequency_cohort", observed=True).agg(
    members=("member_id", "size"), total_spend=("total_spend", "sum"),
    purchases=("number_of_purchases", "sum"), reward_cost=("reward_cost", "sum"))
spend_by_frequency_cohort_df["share_of_members"] = spend_by_frequency_cohort_df["members"] / total_members
spend_by_frequency_cohort_df["share_of_total_spend"] = spend_by_frequency_cohort_df["total_spend"] / total_member_spend
spend_by_frequency_cohort_df["spend_per_member"] = spend_by_frequency_cohort_df["total_spend"] / spend_by_frequency_cohort_df["members"]
spend_by_frequency_cohort_df["reward_as_share_of_spend"] = (spend_by_frequency_cohort_df["reward_cost"]
                                                            / spend_by_frequency_cohort_df["total_spend"].replace(0, np.nan))

spend_by_profile_completeness_df = member_summary_df.groupby("has_incomplete_profile").agg(
    members=("member_id", "size"),
    purchasing_member_rate=("number_of_purchases", lambda purchases: (purchases > 0).mean()),
    spend_per_member=("total_spend", "mean"))
spend_by_profile_completeness_df.index = spend_by_profile_completeness_df.index.map(
    {False: "Complete profile", True: "Incomplete profile (age 118)"})

comparison_tables = [
    ("OFFER PERFORMANCE BY OFFER TYPE", readable_offer_performance(performance_by_offer_type_df)),
    ("OFFER PERFORMANCE BY INDIVIDUAL OFFER", readable_offer_performance(performance_by_offer_df)),
    ("BOGO + DISCOUNT PERFORMANCE BY OFFER-ENGAGEMENT COHORT", readable_offer_performance(performance_by_engagement_cohort_df)),
    ("BOGO + DISCOUNT PERFORMANCE BY PURCHASE-FREQUENCY COHORT", readable_offer_performance(performance_by_frequency_cohort_df)),
    ("VIEW RATE: OFFERS ADVERTISED ON SOCIAL vs NOT (association only, not proof of cause)",
     readable_table(view_rate_by_social_channel_df, percent_columns=["view_rate_not_on_social", "view_rate_on_social"])),
    ("MEMBER SPEND BY PURCHASE-FREQUENCY COHORT",
     readable_table(spend_by_frequency_cohort_df,
                    percent_columns=["share_of_members", "share_of_total_spend", "reward_as_share_of_spend"],
                    money_columns=["total_spend", "reward_cost", "spend_per_member"],
                    count_columns=["members", "purchases"])),
    ("MEMBER SPEND BY PROFILE COMPLETENESS",
     readable_table(spend_by_profile_completeness_df, percent_columns=["purchasing_member_rate"],
                    money_columns=["spend_per_member"], count_columns=["members"])),
]
for table_title, comparison_table_df in comparison_tables:
    print(f"\n{table_title}")
    display(comparison_table_df)


OFFER PERFORMANCE BY OFFER TYPE


,offers_received,members_reached,view_rate,completion_rate,completed_without_view_share,reward_cost,reward_cost_per_offer_sent,reward_cost_without_view,avg_spend_during_offer_window
offer_type,,,,,,,,,
bogo,"30,499","14,992",83.2%,51.4%,29.3%,"£113,440.00",£3.72,"£31,045.00",£26.51
discount,"30,543","14,945",70.6%,58.8%,29.9%,"£51,409.00",£1.68,"£17,725.00",£34.97
informational,"15,235","10,547",65.4%,n/a,n/a,£0.00,£0.00,£0.00,£13.80



OFFER PERFORMANCE BY INDIVIDUAL OFFER


offers_received members_reached view_rate  \
offer_type    offer_label                                                 
bogo          BOGO_r10_d10_5d           7,593           6,330     95.5%   
              BOGO_r10_d10_7d           7,658           6,374     87.7%   
              BOGO_r5_d5_5d             7,571           6,262     95.5%   
              BOGO_r5_d5_7d             7,677           6,355     54.4%   
discount      DISC_r2_d10_10d           7,597           6,332     96.8%   
              DISC_r2_d10_7d            7,632           6,285     54.0%   
              DISC_r3_d7_7d             7,646           6,325     96.2%   
              DISC_r5_d20_10d           7,668           6,374     35.6%   
informational INFO_r0_d0_3d             7,618           6,320     80.8%   
              INFO_r0_d0_4d             7,617           6,331     50.0%   

                              completion_rate completed_without_view_share  \
offer_type    offer_label                                                    
bogo          BOGO_r10_d10_5d           43.9%                        16.9%   
              BOGO_r10_d10_7d           48.2%                        28.7%   
              BOGO_r5_d5_5d             56.7%                        17.5%   
              BOGO_r5_d5_7d             56.7%                        51.0%   
discount      DISC_r2_d10_10d           70.2%                        12.3%   
              DISC_r2_d10_7d            52.7%                        46.6%   
              DISC_r3_d7_7d             67.6%                        14.6%   
              DISC_r5_d20_10d           44.9%                        60.5%   
informational INFO_r0_d0_3d               n/a                          n/a   
              INFO_r0_d0_4d               n/a                          n/a   

                              reward_cost reward_cost_per_offer_sent  \
offer_type    offer_label                                              
bogo          BOGO_r10_d10_5d  £33,310.00                      £4.39   
              BOGO_r10_d10_7d  £36,880.00                      £4.82   
              BOGO_r5_d5_5d    £21,480.00                      £2.84   
              BOGO_r5_d5_7d    £21,770.00                      £2.84   
discount      DISC_r2_d10_10d  £10,664.00                      £1.40   
              DISC_r2_d10_7d    £8,050.00                      £1.05   
              DISC_r3_d7_7d    £15,495.00                      £2.03   
              DISC_r5_d20_10d  £17,200.00                      £2.24   
informational INFO_r0_d0_3d         £0.00                      £0.00   
              INFO_r0_d0_4d         £0.00                      £0.00   

                              reward_cost_without_view  \
offer_type    offer_label                                
bogo          BOGO_r10_d10_5d                £5,620.00   
              BOGO_r10_d10_7d               £10,580.00   
              BOGO_r5_d5_5d                  £3,750.00   
              BOGO_r5_d5_7d                 £11,095.00   
discount      DISC_r2_d10_10d                £1,308.00   
              DISC_r2_d10_7d                 £3,752.00   
              DISC_r3_d7_7d                  £2,265.00   
              DISC_r5_d20_10d               £10,400.00   
informational INFO_r0_d0_3d                      £0.00   
              INFO_r0_d0_4d                      £0.00   

                              avg_spend_during_offer_window  
offer_type    offer_label                                    
bogo          BOGO_r10_d10_5d                        £24.72  
              BOGO_r10_d10_7d                        £31.30  
              BOGO_r5_d5_5d                          £24.34  
              BOGO_r5_d5_7d                          £25.65  
discount      DISC_r2_d10_10d                        £47.40  
              DISC_r2_d10_7d                         £26.57  
              DISC_r3_d7_7d                          £32.96  
              DISC_r5_d20_10d                        £33.04  
informatio


BOGO + DISCOUNT PERFORMANCE BY OFFER-ENGAGEMENT COHORT


,offers_received,members_reached,view_rate,completion_rate,completed_without_view_share,reward_cost,reward_cost_per_offer_sent,reward_cost_without_view,avg_spend_during_offer_window
offer_engagement_cohort,,,,,,,,,
E1 rarely views offers (<25%),"1,375",419,13.9%,31.5%,81.1%,"£2,036.00",£1.48,"£1,650.00",£14.63
E2 views some offers (25-74%),"22,576","6,292",55.9%,48.2%,45.8%,"£51,945.00",£2.30,"£23,395.00",£26.63
E3 views most offers (75%+),"37,091","10,217",92.0%,60.2%,20.7%,"£110,868.00",£2.99,"£23,725.00",£33.85


... (4 outputs left out)


MEMBER SPEND BY PURCHASE-FREQUENCY COHORT


,members,total_spend,purchases,reward_cost,share_of_members,share_of_total_spend,spend_per_member,reward_as_share_of_spend
purchase_frequency_cohort,,,,,,,,
F0 no purchases,422,£0.00,0,£0.00,2.5%,0.0%,£0.00,n/a
F1 1-3 purchases,"2,501","£85,494.46","5,861","£11,482.00",14.7%,4.8%,£34.18,13.4%
F2 4-8 purchases,"7,265","£622,796.05","42,887","£64,076.00",42.7%,35.1%,£85.73,10.3%
F3 9+ purchases,"6,812","£1,067,161.46","90,205","£89,291.00",40.1%,60.1%,£156.66,8.4%



MEMBER SPEND BY PROFILE COMPLETENESS


,members,purchasing_member_rate,spend_per_member
has_incomplete_profile,,,
Complete profile,"14,825",97.8%,£117.03
Incomplete profile (age 118),"2,175",95.9%,£18.63


## What the metrics show

**Members and spend.** 97.5% of members (16,578 of 17,000) made at least one purchase, spending £104.44 per member and £12.78 per purchase. Spend is concentrated: members with 9+ purchases are 40.1% of members but produce 60.1% of spend.

**Offer types.** BOGO offers are viewed more often than discounts (83.2% vs 70.6%) but completed less often (51.4% vs 58.8%), and they cost more than twice as much reward per offer sent (£3.72 vs £1.68). BOGO accounts for £113,440 of the £164,849 reward face value.

**Reward paid without a prior view.** 29.6% of completed offers (9,954 of 33,631) were completed before the member viewed them, carrying £48,770 of reward. This is concentrated in three offers:

| Offer | View rate | Completed without view | Reward without view |
|---|---|---|---|
| DISC_r5_d20_10d | 35.6% | 60.5% | £10,400 |
| BOGO_r5_d5_7d | 54.4% | 51.0% | £11,095 |
| DISC_r2_d10_7d | 54.0% | 46.6% | £3,752 |

Together these three offers account for £25,247 of reward paid without a prior view. None of them is advertised on social.

**Channel association.** Offers advertised on social are viewed far more often: BOGO 92.9% vs 54.4%, discount 96.5% vs 44.8%, informational 80.8% vs 50.0%. This is an association only (see limitations).

**Behavioural cohorts.** Members who rarely view offers have 81.1% of their completions without a prior view, compared with 20.7% for members who view most offers. Frequent buyers (F3) complete 71.6% of promotional offers, yet reward is only 8.4% of their spend, compared with 13.4% for F1. The 422 members who never purchased received 1,532 promotional offers and completed none.

**Implication for the decision.** The offer mix contains offers whose reward is often paid to members who had not seen them. These are candidates for redesign or reduction, but only a controlled holdout test can show whether removing them leaves spend unchanged.

In [25]:
# CELL 31 - Reconciliation: proves no rows were lost or double counted
reconciliation_df = pd.DataFrame([
    ("Purchases", f"{total_purchases:,}", f"{int(member_summary_df['number_of_purchases'].sum()):,}",
     "must match exactly"),
    ("Spend", f"£{total_member_spend:,.2f}", f"£{member_summary_df['total_spend'].sum():,.2f}",
     "must match exactly"),
    ("Completions", f"{int((member_events_df['event_type'] == 'offer_completed').sum()):,} events",
     f"{int(offer_instances_df['was_completed'].sum()):,} instances",
     "small gap expected: one purchase can complete two overlapping copies of an offer"),
    ("Reward face value", f"£{recorded_reward_face_value:,.0f} on events",
     f"£{offer_instances_df['reward_cost'].sum():,.0f} on instances",
     f"£{member_events_df.loc[(member_events_df['event_type'] == 'offer_completed') & ~is_exact_duplicate_event, 'reward_paid'].sum():,.0f} "
     "if exact duplicate events were removed"),
], columns=["quantity", "source transcript", "derived tables", "interpretation"]).set_index("quantity")
print("RECONCILIATION - source totals vs the tables built from them")
display(reconciliation_df)

RECONCILIATION - source totals vs the tables built from them


,source transcript,derived tables,interpretation
quantity,,,
Purchases,"138,953","138,953",must match exactly
Spend,"£1,775,451.97","£1,775,451.97",must match exactly
Completions,"33,579 events","33,631 instances",small gap expected: one purchase can complete two overlapping copies of an o...
Reward face value,"£164,676 on events","£164,849 on instances","£162,679 if exact duplicate events were removed"


## Limitations: fact, inference and assumption

**Fact** = directly observed in the data · **Inference** = a reasoned interpretation, not proven · **Assumption** = taken as given for the analysis

### Missingness
| Statement | Label |
|---|---|
| 2,175 members (12.8%) have age 118 with missing gender and income, always together | Fact |
| These members spend £18.63 per member vs £117.03 for complete profiles, although their purchasing rate is similar (95.9% vs 97.8%) | Fact |
| They are probably members who skipped profile set-up and are less engaged, not a random data loss, so results for them should not be imputed from other members | Inference |
| No member's contact consent is recorded, so the data cannot show who may be targeted individually | Fact |

### Event sequences
| Statement | Label |
|---|---|
| 1,158 views (2.0%) fall outside every validity window, i.e. after the offer expired | Fact |
| 1,776 completion events match more than one offer instance, and 5,295 instances arrived while an identical offer was still valid | Fact |
| 397 completion events are exact duplicates | Fact |
| The duplicates are probably one purchase completing two overlapping copies of the same offer, not recording errors; removing them would lower reward by about 1.2% (£164,676 → £162,679) | Inference |
| The event clock is in hours and offer duration in days | Assumption, supported by the send hours (multiples of 24) and all completions falling inside hour-based windows |
| The meaning of event sequences has not been validated by the data manager | Fact (Exhibit 1) |

### Attribution
| Statement | Label |
|---|---|
| Transactions carry no offer id, so no purchase can be linked to a specific offer | Fact |
| Members typically held 2 offers at once (median), so spend during an offer's window cannot be credited to that offer | Fact |
| 29.6% of completions had no prior view, carrying £48,770 of reward | Fact |
| Much of that reward probably went to purchases members would have made anyway | Inference: view tracking may miss some channels |
| Offers advertised on social are viewed more often | Fact (association) |
| Adding social would raise view rates | Inference: advertised channels are not proof of delivery or cause, and the compared offers also differ in duration |
| There is no randomised control group, so none of these metrics measures incremental sales or profit | Fact |
| Behavioural cohorts are defined on the same period they describe, so they are descriptive only; for future use, define them on the previous cycle | Limitation |
| One dataset period represents one future campaign cycle, and 1 dataset unit = £1 | Assumption (Exhibit 4) |
| There is no product margin data, so contribution cannot be calculated from these files | Fact |